In [ ]:
!pip install torch-geometric

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 54.3 MB/s  0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import sys
import copy
import random
import subprocess

try:
    import torch_geometric
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "torch-geometric"])

if not os.path.exists('/content/drive/MyDrive'):
    from google.colab import drive
    drive.mount('/content/drive')

PROJECT_DIR = '/content/drive/MyDrive/Explainable-AML-GAT-ResNet'
DATA_DIR = f"{PROJECT_DIR}/data"
if PROJECT_DIR not in sys.path:
    sys.path.append(PROJECT_DIR)

# Import trực tiếp kiến trúc từ file models/gat_resnet.py
from models.gat_resnet import GATResNet

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch_geometric.data import Data
from torch_geometric.utils import to_undirected
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    average_precision_score,
    matthews_corrcoef,
)

def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Running on: {device}")

# 1. Tiền xử lý dữ liệu Elliptic (Signed Log1p + StandardScaler fit trên t <= 34)
feat_df = pd.read_csv(f"{DATA_DIR}/elliptic_txs_features.csv", header=None)
class_df = pd.read_csv(f"{DATA_DIR}/elliptic_txs_classes.csv")
edge_df = pd.read_csv(f"{DATA_DIR}/elliptic_txs_edgelist.csv")

tx_ids = feat_df[0].values
time_steps = feat_df[1].values
x_raw = feat_df.iloc[:, 2:].values.astype(np.float32)  # 165 features

x_log = np.sign(x_raw) * np.log1p(np.abs(x_raw))
scaler = StandardScaler()
scaler.fit(x_log[time_steps <= 34])
x_norm = scaler.transform(x_log).astype(np.float32)

label_map = {'1': 1, '2': 0, 'unknown': -1, 1: 1, 2: 0}
class_df['y'] = class_df['class'].map(label_map).fillna(-1).astype(np.int64)
merged = pd.DataFrame({'txId': tx_ids}).merge(class_df[['txId', 'y']], on='txId', how='left')
y_all = merged['y'].values

tx_to_idx = {tx: i for i, tx in enumerate(tx_ids)}
src_idx = edge_df['txId1'].map(tx_to_idx).values
dst_idx = edge_df['txId2'].map(tx_to_idx).values

def build_subgraph(mask):
    old_indices = np.where(mask)[0]
    remap = np.full(len(tx_ids), -1, dtype=np.int64)
    remap[old_indices] = np.arange(len(old_indices))

    edge_mask = mask[src_idx] & mask[dst_idx]
    sub_src = remap[src_idx[edge_mask]]
    sub_dst = remap[dst_idx[edge_mask]]
    edge_index = to_undirected(torch.tensor(np.vstack([sub_src, sub_dst]), dtype=torch.long))

    return Data(
        x=torch.tensor(x_norm[old_indices], dtype=torch.float32),
        edge_index=edge_index,
        y=torch.tensor(y_all[old_indices], dtype=torch.long),
        time_step=torch.tensor(time_steps[old_indices], dtype=torch.long),
    )

train_data = build_subgraph(time_steps <= 34).to(device)
test_data = build_subgraph(time_steps >= 35).to(device)

train_labeled = train_data.y != -1
test_labeled = test_data.y != -1

# 2. Huấn luyện & đánh giá đa hạt giống (5 seeds: 0, 1, 2, 3, 4)
def train_and_eval_seed(seed: int):
    set_seed(seed)
    model = GATResNet(in_channels=165, hidden_channels=100, heads=4, out_channels=2, dropout_prob=0.35, use_skip=True).to(device)
    ema_model = copy.deepcopy(model)
    ema_decay = 0.95

    optimizer = optim.AdamW(model.parameters(), lr=0.0015, weight_decay=3e-3)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=20)
    criterion = nn.CrossEntropyLoss()

    best_prauc = 0.0
    best_state = None
    epochs_no_improve = 0

    for epoch in range(400):
        model.train()
        optimizer.zero_grad()
        logits = model(train_data.x, train_data.edge_index)
        loss = criterion(logits[train_labeled], train_data.y[train_labeled])
        loss.backward()
        optimizer.step()

        with torch.no_grad():
            for ema_p, p in zip(ema_model.parameters(), model.parameters()):
                ema_p.mul_(ema_decay).add_(p, alpha=1.0 - ema_decay)
            for ema_b, b in zip(ema_model.buffers(), model.buffers()):
                ema_b.copy_(b)

        ema_model.eval()
        with torch.no_grad():
            probs = ema_model(test_data.x, test_data.edge_index, return_probs=True)[test_labeled, 1].detach().cpu().numpy()
            trues = test_data.y[test_labeled].detach().cpu().numpy()
            prauc = average_precision_score(trues, probs)
            scheduler.step(prauc)

            if epoch >= 80:
                if prauc > best_prauc:
                    best_prauc = prauc
                    best_state = copy.deepcopy(ema_model.state_dict())
                    epochs_no_improve = 0
                else:
                    epochs_no_improve += 1

        if epoch >= 80 and epochs_no_improve >= 80:
            break

    ema_model.load_state_dict(best_state)
    ema_model.eval()
    with torch.no_grad():
        probs = ema_model(test_data.x, test_data.edge_index, return_probs=True)[test_labeled, 1].detach().cpu().numpy()
        trues = test_data.y[test_labeled].detach().cpu().numpy()

    res = {"seed": seed, "pr_auc": average_precision_score(trues, probs)}
    for tau in [0.50, 0.90]:
        preds = (probs >= tau).astype(int)
        tag = f"t{int(tau*100)}"
        res[f"p_{tag}"] = precision_score(trues, preds, zero_division=0)
        res[f"r_{tag}"] = recall_score(trues, preds, zero_division=0)
        res[f"f1_{tag}"] = f1_score(trues, preds, zero_division=0)
        res[f"micro_{tag}"] = f1_score(trues, preds, average='micro')
        res[f"mcc_{tag}"] = matthews_corrcoef(trues, preds)

    if seed == 0:
        torch.save(best_state, f"{PROJECT_DIR}/best_gat_resnet.pt")
    return res

results = []
for s in [0, 1, 2, 3, 4]:
    r = train_and_eval_seed(s)
    results.append(r)
    print(
        f"Seed {s} | PR-AUC: {r['pr_auc']:.4f} | "
        f"[tau=0.50] P={r['p_t50']:.4f} R={r['r_t50']:.4f} F1={r['f1_t50']:.4f} | "
        f"[tau=0.90] P={r['p_t90']:.4f} R={r['r_t90']:.4f} F1={r['f1_t90']:.4f} MCC={r['mcc_t90']:.4f}"
    )

df_res = pd.DataFrame(results)
df_res.to_csv(f"{DATA_DIR}/gat_resnet_5seeds_results.csv", index=False)

print("\n=== SUMMARY ACROSS 5 SEEDS (MEAN ± STD) ===")
for col in ["pr_auc", "p_t50", "r_t50", "f1_t50", "p_t90", "r_t90", "f1_t90", "micro_t90", "mcc_t90"]:
    print(f"{col:10s}: {df_res[col].mean():.4f} ± {df_res[col].std():.4f}")

/usr/local/lib/python3.13/dist-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


Running on: cuda
Seed 0 | PR-AUC: 0.7701 | [tau=0.50] P=0.7476 R=0.7054 F1=0.7259 | [tau=0.90] P=0.8934 R=0.6657 F1=0.7630 MCC=0.7582
Seed 1 | PR-AUC: 0.7437 | [tau=0.50] P=0.8736 R=0.6380 F1=0.7375 | [tau=0.90] P=0.9303 R=0.5919 F1=0.7235 MCC=0.7290
Seed 2 | PR-AUC: 0.7649 | [tau=0.50] P=0.7655 R=0.7082 F1=0.7357 | [tau=0.90] P=0.8916 R=0.6833 F1=0.7737 MCC=0.7677
Seed 3 | PR-AUC: 0.7695 | [tau=0.50] P=0.8485 R=0.6981 F1=0.7660 | [tau=0.90] P=0.9274 R=0.6722 F1=0.7794 MCC=0.7779
Seed 4 | PR-AUC: 0.7550 | [tau=0.50] P=0.8793 R=0.6593 F1=0.7536 | [tau=0.90] P=0.9461 R=0.5993 F1=0.7337 MCC=0.7406

=== SUMMARY ACROSS 5 SEEDS (MEAN ± STD) ===
pr_auc    : 0.7606 ± 0.0112
p_t50     : 0.8229 ± 0.0620
r_t50     : 0.6818 ± 0.0314
f1_t50    : 0.7437 ± 0.0159
p_t90     : 0.9178 ± 0.0241
r_t90     : 0.6425 ± 0.0434
f1_t90    : 0.7547 ± 0.0248
micro_t90 : 0.9730 ± 0.0018
mcc_t90   : 0.7547 ± 0.0198
